In [ ]:
# =====================================================
# IMPORTS
# =====================================================

import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import StackingRegressor

from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

# =====================================================
# LOAD DATASET (ENCODING SAFE)
# =====================================================

encodings = [
    "utf-8",
    "latin1",
    "ISO-8859-1",
    "cp1252"
]

df = None

for enc in encodings:
    try:
        df = pd.read_csv(
            "../data/laptop_price.csv",
            encoding=enc
        )

        print(f"Loaded successfully using {enc}")
        break

    except:
        pass

if df is None:
    raise Exception("Could not read CSV file")

print("\nDataset Shape:")
print(df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 Rows:")
print(df.head())

# =====================================================
# CLEANING
# =====================================================

if "laptop_ID" in df.columns:
    df.drop("laptop_ID", axis=1, inplace=True)

# remove euro symbol if present
if df["Price_euros"].dtype == object:
    df["Price_euros"] = (
        df["Price_euros"]
        .astype(str)
        .str.replace("€", "", regex=False)
        .str.replace(",", "", regex=False)
        .astype(float)
    )

# =====================================================
# FEATURES & TARGET
# =====================================================

X = df.drop("Price_euros", axis=1)
y = df["Price_euros"]

# =====================================================
# TRAIN TEST SPLIT
# =====================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# =====================================================
# PREPROCESSING
# =====================================================

cat_cols = X.select_dtypes(
    include="object"
).columns

num_cols = X.select_dtypes(
    exclude="object"
).columns

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            cat_cols
        ),
        (
            "num",
            StandardScaler(),
            num_cols
        )
    ]
)

# =====================================================
# BASE LEARNERS
# =====================================================

lr = LinearRegression()

dt = DecisionTreeRegressor(
    max_depth=8,
    random_state=42
)

rf = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

# =====================================================
# INDIVIDUAL MODEL COMPARISON
# =====================================================

models = {
    "Linear Regression": lr,
    "Decision Tree": dt,
    "Random Forest": rf
}

results = {}

print("\n========== BASE LEARNERS ==========\n")

for name, model in models.items():

    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipe.fit(X_train, y_train)

    pred = pipe.predict(X_test)

    r2 = r2_score(
        y_test,
        pred
    )

    results[name] = r2

    print(
        f"{name}: {r2:.4f}"
    )

# =====================================================
# STACKING REGRESSOR
# =====================================================

estimators = [
    ("lr", lr),
    ("dt", dt),
    ("rf", rf)
]

stack_reg = StackingRegressor(
    estimators=estimators,
    final_estimator=LinearRegression()
)

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", stack_reg)
])

# =====================================================
# TRAIN STACKING MODEL
# =====================================================

pipeline.fit(
    X_train,
    y_train
)

# =====================================================
# PREDICTIONS
# =====================================================

pred = pipeline.predict(
    X_test
)

# =====================================================
# EVALUATION
# =====================================================

r2 = r2_score(
    y_test,
    pred
)

mae = mean_absolute_error(
    y_test,
    pred
)

mse = mean_squared_error(
    y_test,
    pred
)

rmse = np.sqrt(mse)

results["Stacking Regressor"] = r2

print("\n========== STACKING RESULTS ==========\n")

print("R2 Score :", round(r2, 4))
print("MAE      :", round(mae, 2))
print("MSE      :", round(mse, 2))
print("RMSE     :", round(rmse, 2))

# =====================================================
# COMPARISON TABLE
# =====================================================

comparison_df = pd.DataFrame({
    "Model": results.keys(),
    "R2 Score": results.values()
})

print("\n========== MODEL COMPARISON ==========\n")

print(comparison_df)

# =====================================================
# VISUALIZATION
# =====================================================

plt.figure(figsize=(7,4))

bars = plt.bar(
    comparison_df["Model"],
    comparison_df["R2 Score"]
)

plt.title(
    "Stacking Regression Performance Comparison"
)

plt.ylabel("R2 Score")

for bar in bars:

    height = bar.get_height()

    plt.text(
        bar.get_x() + bar.get_width()/2,
        height,
        f"{height:.3f}",
        ha="center"
    )

plt.tight_layout()
plt.show()

# =====================================================
# SAVE MODEL
# =====================================================

pickle.dump(
    pipeline,
    open(
        "../models/stacking_regressor.pkl",
        "wb"
    )
)

print(
    "\nStacking Regressor Saved Successfully"
)

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xe9 in position 131543: invalid continuation byte